# Обучение SegFormer

Полный Kaggle-пайплайн: два источника, разные ROI, PPT phase maps, аугментация, обучение и метрики. Перед Run All включите GPU и Internet в Kaggle Settings.

In [ ]:
# Версии аугментации точно взяты из requirements.txt ветки main.
import subprocess, sys, os
from pathlib import Path

PINS = [
    'albumentations==2.0.8', 'opencv-python-headless==5.0.0.93',
    'scikit-learn==1.9.0', 'PyYAML==6.0.3',
    'matplotlib==3.11.1', 'requests==2.34.2',
    'transformers==5.15.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PINS])

REPO_DIR = Path('/kaggle/working/thermal-control-ya-project')
if not (REPO_DIR / '.git').exists():
    subprocess.check_call([
        'git', 'clone', '--depth', '1', '--branch', 'segformer',
        '--single-branch',
        'https://github.com/tomatoCoderq/thermal-control-ya-project.git',
        str(REPO_DIR),
    ])
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

In [ ]:
import torch, albumentations, cv2, transformers
print('torch', torch.__version__, '| albumentations', albumentations.__version__)
print('opencv', cv2.__version__, '| transformers', transformers.__version__)
assert torch.cuda.is_available(), 'Включите GPU в Kaggle Settings'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Находим Kaggle dataset при любой вложенности указанного пути.
from collections import Counter

def common_parent(paths):
    return Counter(p.parent for p in paths).most_common(1)[0][0] if paths else None

INPUT_ROOT = Path('/kaggle/input')
all_mats = sorted(INPUT_ROOT.rglob('*.mat'))
kaggle_mats = [p for p in all_mats if 'irt-pvc-depth' in str(p).lower()] or all_mats
kaggle_masks = sorted(
    p for p in INPUT_ROOT.rglob('*.png')
    if ('mask' in str(p).lower() or 'label' in str(p).lower())
    and ('irt-pvc-depth' in str(p).lower())
)
if not kaggle_masks:
    kaggle_masks = sorted(
        p for p in INPUT_ROOT.rglob('*.png')
        if 'mask' in str(p).lower() or 'label' in str(p).lower()
    )
assert kaggle_mats, 'Не найдены MAT: добавьте ziangwei/irt-pvc-depth через Add Input'
assert kaggle_masks, 'Не найдены PNG-маски Kaggle'
KAGGLE_DATA_DIR = common_parent(kaggle_mats)
KAGGLE_MASK_DIR = common_parent(kaggle_masks)
print('Kaggle:', len(kaggle_mats), 'videos |', len(kaggle_masks), 'masks')
print(KAGGLE_DATA_DIR, KAGGLE_MASK_DIR)

## Датасет Яндекс

Ячейка ниже скачивает публичную папку в /kaggle/working/yandex_data. Если личный augmentation notebook уже скачал её, существующие файлы пропускаются. Без сегментационных PNG-масок Yandex-видео не добавляются в supervised loss.

In [ ]:
import requests, zipfile

YANDEX_URL = 'https://disk.yandex.ru/d/POr5765WUdKLbg'
YANDEX_DIR = Path('/kaggle/working/yandex_data')
DOWNLOAD_YANDEX = True
API = 'https://cloud-api.yandex.net/v1/disk/public/resources'

def download_file(url, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size > 0:
        return
    with requests.get(url, stream=True, timeout=180) as response:
        response.raise_for_status()
        with path.open('wb') as output:
            for chunk in response.iter_content(8 * 1024 * 1024):
                if chunk: output.write(chunk)
    print('Скачан:', path.name)

def resource(public_key, remote_path=None):
    params = {'public_key': public_key, 'limit': 1000}
    if remote_path is not None: params['path'] = remote_path
    answer = requests.get(API, params=params, timeout=120)
    answer.raise_for_status()
    return answer.json()

def download_tree(public_key, local_dir, remote_path=None):
    item = resource(public_key, remote_path)
    if item['type'] == 'file':
        download_file(item['file'], local_dir / item['name']); return
    local_dir.mkdir(parents=True, exist_ok=True)
    for child in item.get('_embedded', {}).get('items', []):
        target = local_dir / child['name']
        if child['type'] == 'dir':
            download_tree(public_key, target, child['path'])
        else:
            download_file(child['file'], target)

if DOWNLOAD_YANDEX: download_tree(YANDEX_URL, YANDEX_DIR)
for archive in YANDEX_DIR.rglob('*.zip') if YANDEX_DIR.exists() else []:
    target = archive.with_suffix('')
    if not target.exists():
        target.mkdir(parents=True)
        with zipfile.ZipFile(archive) as zf: zf.extractall(target)

yandex_mats = sorted(YANDEX_DIR.rglob('*.mat')) if YANDEX_DIR.exists() else []
yandex_masks = sorted(
    p for p in YANDEX_DIR.rglob('*.png')
    if 'mask' in str(p).lower() or 'label' in str(p).lower()
) if YANDEX_DIR.exists() else []
YANDEX_DATA_DIR = common_parent(yandex_mats)
YANDEX_MASK_DIR = common_parent(yandex_masks)
print('Yandex:', len(yandex_mats), 'videos |', len(yandex_masks), 'segmentation masks')

In [ ]:
# МЕНЯТЬ ROI И ПАРАМЕТРЫ ОБУЧЕНИЯ НУЖНО ЗДЕСЬ.
KAGGLE_ROI = {'x': 80, 'y': 25, 'w': 178, 'h': 190}
YANDEX_ROI = {'x': 80, 'y': 43, 'w': 178, 'h': 120}
PER_VIDEO_ROIS = {
    # 'R_002': {'x': 82, 'y': 27, 'w': 175, 'h': 185},
    # 'Sample_20_Static': {'x': 78, 'y': 44, 'w': 180, 'h': 118},
}
EPOCHS = 30
BATCH_SIZE = 4
LEARNING_RATE = 6e-5
GRID_SHUFFLE_P = 0.0  # потом отдельно сравнить с 0.15
MODEL_NAME = 'nvidia/mit-b0'

In [ ]:
from copy import deepcopy
import yaml

template = yaml.safe_load(Path('configs/augmentation_ppt.yaml').read_text())
dataset_cfg = deepcopy(template['dataset'])
sources = [{
    'root': str(KAGGLE_DATA_DIR), 'masks': str(KAGGLE_MASK_DIR),
    'pattern': '*.mat', 'object_roi': KAGGLE_ROI,
}]
if YANDEX_DATA_DIR is not None and YANDEX_MASK_DIR is not None:
    sources.append({
        'root': str(YANDEX_DATA_DIR), 'masks': str(YANDEX_MASK_DIR),
        'pattern': '*.mat', 'object_roi': YANDEX_ROI,
    })
elif YANDEX_DATA_DIR is not None:
    print('Yandex пропущен: нужны pixel-wise PNG-маски, меток глубины недостаточно.')

dataset_cfg['sources'] = sources
dataset_cfg['files_meta'] = {k: {'object_roi': v} for k, v in PER_VIDEO_ROIS.items()}
dataset_cfg['cache_dir'] = '/kaggle/working/thermal/cache'
dataset_cfg['features']['cache_dir'] = '/kaggle/working/thermal/ppt_features'
dataset_cfg['loader']['batch_size'] = BATCH_SIZE
dataset_cfg['loader']['num_workers'] = 2
for spec in dataset_cfg['augs']['spatial']:
    if spec['name'] == 'RandomGridShuffle': spec['params']['p'] = GRID_SHUFFLE_P

training_cfg = {
    'model_name': MODEL_NAME, 'pretrained': True, 'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE, 'weight_decay': 0.01, 'dice_weight': 0.5,
    'val_fraction': 0.15, 'test_fraction': 0.15, 'split_seed': 42,
    'amp': True, 'precompute_features': True,
    'output_dir': '/kaggle/working/thermal/segformer',
}
RUN_CONFIG = Path('/kaggle/working/segformer_run.yaml')
RUN_CONFIG.write_text(
    yaml.safe_dump({'dataset': dataset_cfg, 'training': training_cfg}, sort_keys=False),
    encoding='utf-8',
)
print('Источников:', len(sources), '| config:', RUN_CONFIG)

In [ ]:
# Sanity check одного видео перед долгим запуском.
from irt_data.config import DatasetConfig
from irt_data.dataset import IRTDataset

check = deepcopy(dataset_cfg)
check['train'] = False
check['samples_per_video'] = 1
check['sources'] = [deepcopy(sources[0])]
check['sources'][0]['pattern'] = kaggle_mats[0].name
sample = IRTDataset(DatasetConfig.from_dict(check))[0]
print('image', tuple(sample['image'].shape), '| mask', tuple(sample['mask'].shape))
print('video', sample['video_id'], '| classes', sample['mask'].unique().tolist())
assert tuple(sample['image'].shape) == (3, 256, 256)
assert tuple(sample['mask'].shape) == (256, 256)
assert set(sample['mask'].unique().tolist()) <= {0, 1}

In [ ]:
# ДОЛГАЯ ЯЧЕЙКА: PPT cache и обучение.
LOG = Path('/kaggle/working/thermal/segformer/train.log')
LOG.parent.mkdir(parents=True, exist_ok=True)
command = [sys.executable, 'train_segformer.py', '--config', str(RUN_CONFIG)]
with LOG.open('w', encoding='utf-8') as log_file:
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end=''); log_file.write(line)
    code = process.wait()
if code != 0: raise RuntimeError(f'Ошибка обучения, см. {LOG}')

In [ ]:
import json, re, shutil
import matplotlib.pyplot as plt

OUTPUT = Path(training_cfg['output_dir'])
history_path = OUTPUT / 'history.json'
if history_path.exists():
    history = json.loads(history_path.read_text())
else:
    # Совместимость с уже запушенной версией train_segformer.py: читаем epoch-строки log.
    pattern = re.compile(r'epoch=(\d+) train_loss=([0-9.]+) val_loss=([0-9.]+) dice=([0-9.]+) iou=([0-9.]+)')
    history = []
    for line in LOG.read_text().splitlines():
        match = pattern.search(line)
        if match:
            epoch, train_loss, val_loss, dice, iou = match.groups()
            history.append({
                'epoch': int(epoch), 'train_loss': float(train_loss),
                'val_loss': float(val_loss), 'val_dice': float(dice),
                'val_iou': float(iou),
            })
assert history, f'История эпох не найдена в {history_path} или {LOG}'
metrics = json.loads((OUTPUT / 'test_metrics.json').read_text())
epochs = [r['epoch'] for r in history]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, [r['train_loss'] for r in history], label='train')
axes[0].plot(epochs, [r['val_loss'] for r in history], label='val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid()
axes[1].plot(epochs, [r['val_dice'] for r in history], label='Dice')
axes[1].plot(epochs, [r['val_iou'] for r in history], label='IoU')
axes[1].set_title('Validation'); axes[1].legend(); axes[1].grid()
plt.show()
print('TEST:', json.dumps(metrics, indent=2))
archive = shutil.make_archive('/kaggle/working/segformer_results', 'zip', root_dir=OUTPUT)
print('Best checkpoint:', OUTPUT / 'best.pt')
print('Скачайте архив:', archive)